1. Create Skewed dataset for testing purposes

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand, floor
import random

spark = SparkSession.builder.appName("LargeSkewedData").getOrCreate()

# Control size and skew
total_rows = 50_000_000               # 50M rows (change as needed)
num_normal_customers = 1_000_000       # Normal customers
skewed_customer_base = 1000001         # Base ID for skewed customers
num_skewed_customers = 5               # 5 super‑heavy customers

# Generate IDs with skew
skewed_ratio = 0.6  # e.g., 60% of records belong to the skewed customers

skewed_end = int(total_rows * skewed_ratio)
normal_end = total_rows

df = spark.range(normal_end).select(
    col("id").alias("row_id")
).withColumn(
    "r", rand(12345)
).withColumn(
    "customer_id",
    # Skewed bucket: 60% of rows go into 5 large customers
    when(col("row_id") < skewed_end, floor(col("r") * num_skewed_customers) + skewed_customer_base)
    # Normal: spread over ~1M customers
    .otherwise(floor(col("r") * num_normal_customers) + 1)
).select(
    col("customer_id"),
    (col("row_id") + 10000000).cast("string").alias("order_id"),
    (rand(1234) * 1000).alias("amount"),
    (rand(1234) * 10).cast("int").alias("city_id")
)

# Quick check: show skew
df.groupBy("customer_id").count().orderBy("count", ascending=False).show(10)

# Save to disk for repeated testing
df.write.mode("overwrite").parquet("large_skewed_customers.parquet")

+-----------+-------+
|customer_id|  count|
+-----------+-------+
|    1000001|6002487|
|    1000003|6001478|
|    1000005|5999647|
|    1000004|5999319|
|    1000002|5997069|
|     847869|     45|
|     829948|     44|
|     612204|     44|
|      15206|     43|
|     198392|     43|
+-----------+-------+
only showing top 10 rows


2. Testing Spark repartitions

In [8]:
from pyspark.sql.functions import spark_partition_id

# spark repartitioning

# Example: Repartition by customer_id to handle skew
repartitioned_df = df.repartition(8)

print(repartitioned_df.rdd.getNumPartitions())

repartitioned_df.withColumn("partition_id", spark_partition_id()).groupBy("partition_id").count().orderBy("count", ascending=False).show()

8
+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|6250000|
|           1|6250000|
|           2|6250000|
|           3|6250000|
|           4|6250000|
|           5|6250000|
|           6|6250000|
|           7|6250000|
+------------+-------+

